##### Importing and connecting

In [8]:
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import text

# Leave False until the source audit and target preview have been reviewed.
APPLY_USER_MIGRATION = True
SOURCE_SYSTEM = "risk"

BACKEND_DIR = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(BACKEND_DIR / "migrations"))
from common import legacy_engine, new_engine  # noqa: E402

##### Extract legacy users (read-only)

In [9]:
with legacy_engine.connect() as conn:
    with conn.begin():
        conn.execute(text("SET TRANSACTION READ ONLY"))
        legacy_users = pd.read_sql(
            text("""
            SELECT id, uname, platform, join_date, discord_id
            FROM public.users
            ORDER BY id
        """),
            conn,
        )

print(f"Legacy user count: {len(legacy_users):,}")
display(legacy_users.head(25))

Legacy user count: 38,117


,id,uname,platform,join_date,discord_id
0,1,bakonydraco,reddit,2022-06-06 23:44:23.557466,NaN
1,2,cfbflair,reddit,2022-06-06 23:44:23.557466,NaN
2,3,StrikerObi,reddit,2022-06-06 23:44:23.557466,NaN
3,4,A-Stu-Ute,reddit,2022-06-06 23:44:23.557466,NaN
4,5,I_am_bot_beep_boop,reddit,2022-06-06 23:44:23.557466,NaN
5,6,xelphin,reddit,2022-06-06 23:44:23.557466,NaN
6,7,jayhawx19,reddit,2022-06-06 23:44:23.557466,NaN
7,8,Qurtys_Lyn,reddit,2022-06-06 23:44:23.557466,NaN
8,9,CSU_Mike,reddit,2022-06-06 23:44:23.557466,NaN
9,10,SometimesY,reddit,2022-06-06 23:44:23.557466,NaN


##### Audit source data

In [10]:
source_audit = pd.Series(
    {
        "row_count": len(legacy_users),
        "distinct_ids": legacy_users["id"].nunique(dropna=True),
        "null_ids": legacy_users["id"].isna().sum(),
        "null_usernames": legacy_users["uname"].isna().sum(),
        "blank_usernames": legacy_users["uname"].astype("string").str.strip().eq("").sum(),
        "null_platforms": legacy_users["platform"].isna().sum(),
        "null_join_dates": legacy_users["join_date"].isna().sum(),
        "users_with_discord_id": legacy_users["discord_id"].notna().sum(),
        "distinct_discord_ids": legacy_users["discord_id"].nunique(dropna=True),
    }
)
display(source_audit.to_frame("value"))
display(legacy_users["platform"].value_counts(dropna=False).rename_axis("platform").reset_index(name="count"))

duplicate_legacy_ids = legacy_users.loc[legacy_users["id"].duplicated(keep=False)].sort_values("id")
discord_rows = legacy_users.loc[legacy_users["discord_id"].notna()].copy()
duplicate_discord_ids = discord_rows.loc[discord_rows["discord_id"].duplicated(keep=False)].sort_values("discord_id")
display(duplicate_legacy_ids)
display(duplicate_discord_ids)

,value
row_count,38117
distinct_ids,38117
null_ids,0
null_usernames,0
blank_usernames,0
null_platforms,0
null_join_dates,0
users_with_discord_id,97
distinct_discord_ids,97


,platform,count
0,reddit,37836
1,discord,257
2,deleted,24


,id,uname,platform,join_date,discord_id


,id,uname,platform,join_date,discord_id


##### Transform legacy users and Discord identities

In [11]:
# Keep source metadata available for auditing; only fields supported by the
# current destination schema are written to users / user_auth / mappings.
users_to_migrate = legacy_users[["id", "uname", "platform", "join_date", "discord_id"]].copy()
users_to_migrate["legacy_user_id"] = users_to_migrate["id"].astype("int64")
users_to_migrate["display_name"] = users_to_migrate["uname"].astype("string").str.strip()
users_to_migrate["display_name"] = users_to_migrate["display_name"].replace("", pd.NA)

# Discord snowflake IDs are strings in user_auth.provider_subject.
# Do not infer Reddit identities from usernames or platform labels.
discord_identities = users_to_migrate.loc[
    users_to_migrate["discord_id"].notna(), ["legacy_user_id", "discord_id"]
].copy()
discord_identities["provider"] = "discord"
discord_identities["provider_subject"] = discord_identities["discord_id"].map(lambda value: str(int(value)))
discord_identities = discord_identities[["legacy_user_id", "provider", "provider_subject"]]

display(users_to_migrate[["legacy_user_id", "display_name", "platform", "join_date", "discord_id"]].head(25))
display(discord_identities.head(25))
print(f"Users prepared: {len(users_to_migrate):,}")
print(f"Discord identities prepared: {len(discord_identities):,}")

,legacy_user_id,display_name,platform,join_date,discord_id
0,1,bakonydraco,reddit,2022-06-06 23:44:23.557466,NaN
1,2,cfbflair,reddit,2022-06-06 23:44:23.557466,NaN
2,3,StrikerObi,reddit,2022-06-06 23:44:23.557466,NaN
3,4,A-Stu-Ute,reddit,2022-06-06 23:44:23.557466,NaN
4,5,I_am_bot_beep_boop,reddit,2022-06-06 23:44:23.557466,NaN
5,6,xelphin,reddit,2022-06-06 23:44:23.557466,NaN
6,7,jayhawx19,reddit,2022-06-06 23:44:23.557466,NaN
7,8,Qurtys_Lyn,reddit,2022-06-06 23:44:23.557466,NaN
8,9,CSU_Mike,reddit,2022-06-06 23:44:23.557466,NaN
9,10,SometimesY,reddit,2022-06-06 23:44:23.557466,NaN


,legacy_user_id,provider,provider_subject
5834,5835,discord,298062706123210752
8025,8026,discord,137941827171909632
8907,8908,discord,446334033300029440
11579,11580,discord,328687799949852672
14662,16970,discord,455954988741427200
19269,21577,discord,254103481634258944
20456,22764,discord,815280829969596416
21338,23646,discord,264165243851243520
22734,25042,discord,135864029036937216
29031,33143,discord,374688755128336384


Users prepared: 38,117
Discord identities prepared: 97


##### Validate target schema and migration assumptions

In [12]:
required_columns = {
    "users": {"id", "display_name", "is_admin", "is_active", "keycloak_subject", "merged_into_user_id"},
    "user_auth": {"id", "user_id", "provider", "provider_subject", "provider_username", "linked_at"},
    "legacy_user_mappings": {"id", "source_system", "legacy_user_id", "user_id", "migrated_at"},
}
target_schema = {}
with new_engine.connect() as conn:
    for table_name, expected_columns in required_columns.items():
        columns = set(
            conn.execute(
                text("""
            SELECT column_name FROM information_schema.columns
            WHERE table_schema = 'public' AND table_name = :table_name
        """),
                {"table_name": table_name},
            )
            .scalars()
            .all()
        )
        target_schema[table_name] = sorted(columns)
        missing = expected_columns - columns
        if missing:
            raise RuntimeError(f"public.{table_name} is missing columns: {sorted(missing)}")
display(pd.DataFrame([{"table": table, "columns": ", ".join(columns)} for table, columns in target_schema.items()]))

issues = []
if legacy_users["id"].isna().any():
    issues.append("Legacy user IDs contain NULL values.")
if legacy_users["id"].duplicated().any():
    issues.append("Legacy user IDs are duplicated.")
if discord_rows["discord_id"].duplicated().any():
    issues.append("Discord IDs are duplicated in legacy users.")
if (users_to_migrate["legacy_user_id"] < 0).any():
    issues.append("Unexpected negative legacy user ID found.")
if issues:
    display(pd.DataFrame({"issue": issues}))
    raise ValueError("Resolve source validation issues before migration.")
print("Source validation passed.")

,table,columns
0,users,"created_at, display_name, id, is_active, is_ad..."
1,user_auth,"id, linked_at, provider, provider_subject, pro..."
2,legacy_user_mappings,"id, legacy_user_id, migrated_at, source_system..."


Source validation passed.


##### Preview existing target state

In [13]:
with new_engine.connect() as conn:
    target_counts = (
        conn.execute(
            text("""
        SELECT 'users' AS table_name, COUNT(*) AS row_count FROM public.users
        UNION ALL SELECT 'legacy_user_mappings', COUNT(*) FROM public.legacy_user_mappings
        UNION ALL SELECT 'user_auth', COUNT(*) FROM public.user_auth
    """)
        )
        .mappings()
        .all()
    )
    existing_mappings = pd.read_sql(
        text("""
        SELECT source_system, legacy_user_id, user_id
        FROM public.legacy_user_mappings
        WHERE source_system = :source_system
        ORDER BY legacy_user_id
    """),
        conn,
        params={"source_system": SOURCE_SYSTEM},
    )
display(pd.DataFrame(target_counts))
display(existing_mappings.head(25))

existing_legacy_ids = set(existing_mappings["legacy_user_id"].astype("int64"))
source_legacy_ids = set(users_to_migrate["legacy_user_id"].astype("int64"))
missing_legacy_ids = sorted(source_legacy_ids - existing_legacy_ids)
unexpected_legacy_ids = sorted(existing_legacy_ids - source_legacy_ids)
print(f"Already mapped: {len(existing_legacy_ids & source_legacy_ids):,}")
print(f"Not yet mapped: {len(missing_legacy_ids):,}")
print(f"Mappings absent from current source: {len(unexpected_legacy_ids):,}")
if unexpected_legacy_ids:
    display(existing_mappings.loc[existing_mappings["legacy_user_id"].isin(unexpected_legacy_ids)])

,table_name,row_count
0,users,0
1,legacy_user_mappings,0
2,user_auth,0


,source_system,legacy_user_id,user_id


Already mapped: 0
Not yet mapped: 38,117
Mappings absent from current source: 0


##### Writing to SQL — opt-in, transactional, rerunnable

In [14]:
if not APPLY_USER_MIGRATION:
    raise RuntimeError(
        "Import disabled. Review source audit and target preview, then set "
        "APPLY_USER_MIGRATION = True in the first cell."
    )

# Do not mix a fresh import with unrelated manually-created users.
with new_engine.begin() as conn:
    target_user_count = conn.execute(text("SELECT COUNT(*) FROM public.users")).scalar_one()
    mapped_count = conn.execute(
        text("""
        SELECT COUNT(*) FROM public.legacy_user_mappings WHERE source_system = :source_system
    """),
        {"source_system": SOURCE_SYSTEM},
    ).scalar_one()
    if target_user_count > 0 and mapped_count == 0:
        raise RuntimeError(
            f"Target already has {target_user_count} users but no '{SOURCE_SYSTEM}' mappings. "
            "Reconcile target before importing; no changes were committed."
        )

    mapping_rows = (
        conn.execute(
            text("""
        SELECT legacy_user_id, user_id FROM public.legacy_user_mappings
        WHERE source_system = :source_system FOR UPDATE
    """),
            {"source_system": SOURCE_SYSTEM},
        )
        .mappings()
        .all()
    )
    legacy_to_user = {int(row["legacy_user_id"]): int(row["user_id"]) for row in mapping_rows}

    broken_ids = (
        conn.execute(
            text("""
        SELECT m.legacy_user_id FROM public.legacy_user_mappings m
        LEFT JOIN public.users u ON u.id = m.user_id
        WHERE m.source_system = :source_system AND u.id IS NULL
    """),
            {"source_system": SOURCE_SYSTEM},
        )
        .scalars()
        .all()
    )
    if broken_ids:
        raise RuntimeError(f"Mappings point to missing users: {broken_ids[:20]}")

    insert_user = text("""
        INSERT INTO public.users (display_name, is_admin, is_active)
        VALUES (:display_name, FALSE, TRUE) RETURNING id
    """)
    insert_mapping = text("""
        INSERT INTO public.legacy_user_mappings (source_system, legacy_user_id, user_id)
        VALUES (:source_system, :legacy_user_id, :user_id)
    """)
    insert_identity = text("""
        INSERT INTO public.user_auth (user_id, provider, provider_subject)
        VALUES (:user_id, 'discord', :provider_subject)
    """)

    created_users = created_mappings = created_discord_identities = 0
    for row in users_to_migrate.to_dict("records"):
        legacy_id = int(row["legacy_user_id"])
        user_id = legacy_to_user.get(legacy_id)
        if user_id is None:
            display_name = None if pd.isna(row["display_name"]) else str(row["display_name"])
            user_id = conn.execute(insert_user, {"display_name": display_name}).scalar_one()
            conn.execute(
                insert_mapping,
                {
                    "source_system": SOURCE_SYSTEM,
                    "legacy_user_id": legacy_id,
                    "user_id": user_id,
                },
            )
            legacy_to_user[legacy_id] = user_id
            created_users += 1
            created_mappings += 1

        if pd.isna(row["discord_id"]):
            continue
        discord_subject = str(int(row["discord_id"]))
        identity_owner = conn.execute(
            text("""
            SELECT user_id FROM public.user_auth
            WHERE provider = 'discord' AND provider_subject = :subject
        """),
            {"subject": discord_subject},
        ).scalar_one_or_none()
        if identity_owner is not None:
            if int(identity_owner) != user_id:
                raise RuntimeError(
                    f"Discord ID {discord_subject} belongs to CFBR user {identity_owner}, "
                    f"but legacy user {legacy_id} maps to CFBR user {user_id}."
                )
            continue

        existing_discord_for_user = conn.execute(
            text("""
            SELECT provider_subject FROM public.user_auth
            WHERE user_id = :user_id AND provider = 'discord'
        """),
            {"user_id": user_id},
        ).scalar_one_or_none()
        if existing_discord_for_user is not None:
            raise RuntimeError(f"CFBR user {user_id} already has a different Discord identity.")
        conn.execute(insert_identity, {"user_id": user_id, "provider_subject": discord_subject})
        created_discord_identities += 1

    final_mapping_count = conn.execute(
        text("""
        SELECT COUNT(*) FROM public.legacy_user_mappings WHERE source_system = :source_system
    """),
        {"source_system": SOURCE_SYSTEM},
    ).scalar_one()
    final_broken_count = conn.execute(
        text("""
        SELECT COUNT(*) FROM public.legacy_user_mappings m
        LEFT JOIN public.users u ON u.id = m.user_id
        WHERE m.source_system = :source_system AND u.id IS NULL
    """),
        {"source_system": SOURCE_SYSTEM},
    ).scalar_one()
    final_admin_count = conn.execute(text("SELECT COUNT(*) FROM public.users WHERE is_admin = TRUE")).scalar_one()
    if final_mapping_count != len(users_to_migrate):
        raise RuntimeError(f"Expected {len(users_to_migrate)} mappings, found {final_mapping_count}.")
    if final_broken_count != 0:
        raise RuntimeError(f"Found {final_broken_count} mappings to missing users.")
    if final_admin_count != 0:
        raise RuntimeError(f"Expected zero admins after migration, found {final_admin_count}.")

print(f"New users created: {created_users:,}")
print(f"Legacy mappings created: {created_mappings:,}")
print(f"Discord identities created: {created_discord_identities:,}")
print(f"Total legacy mappings verified: {final_mapping_count:,}")

New users created: 38,117
Legacy mappings created: 38,117
Discord identities created: 97
Total legacy mappings verified: 38,117


##### Post-migration verification

Run after the write cell. This verifies mappings and identity counts; it does not migrate historical game tables.

In [15]:
with new_engine.connect() as conn:
    verification = pd.read_sql(
        text("""
        SELECT
          (SELECT COUNT(*) FROM public.legacy_user_mappings WHERE source_system = :source_system) AS mapping_count,
          (SELECT COUNT(*) FROM public.legacy_user_mappings m JOIN public.users u ON u.id = m.user_id
           WHERE m.source_system = :source_system) AS mapped_user_count,
          (SELECT COUNT(*) FROM public.legacy_user_mappings m LEFT JOIN public.users u ON u.id = m.user_id
           WHERE m.source_system = :source_system AND u.id IS NULL) AS broken_mappings,
          (SELECT COUNT(*) FROM public.user_auth WHERE provider = 'discord') AS discord_identity_count,
          (SELECT COUNT(*) FROM public.users WHERE is_admin) AS admin_count
    """),
        conn,
        params={"source_system": SOURCE_SYSTEM},
    )
display(verification)

if int(verification.loc[0, "mapping_count"]) != len(legacy_users):
    raise ValueError("Legacy mapping count does not match the extracted source count.")
if int(verification.loc[0, "broken_mappings"]) != 0:
    raise ValueError("One or more mappings reference missing users.")
print("Post-migration verification passed.")

,mapping_count,mapped_user_count,broken_mappings,discord_identity_count,admin_count
0,38117,38117,0,97,0


Post-migration verification passed.
